In [1]:
import numpy as np
import pandas as pd

C:\Users\GEOMAB\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.11.0' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import (
C:\Users\GEOMAB\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


In [2]:
df = pd.read_csv('World cup Train.csv')

In [11]:
df.columns

Index(['ID', 'team_id', 'country', 'team_code', 'confederation_name',
       'region_name', 'tournament_id', 'tournament_name', 'year',
       'matches_played', 'total_goals', 'stage_reached'],
      dtype='str')

In [4]:
import numpy as np
import pandas as pd

In [11]:
df = pd.read_csv('World cup Train.csv')

In [17]:
# ==============================================================================
# Step 1: 
# ==============================================================================

# 1. Load the training dataset
df = pd.read_csv("World cup Train.csv")

# 2. Check the size of the data (rows, columns)
print("Dataset shape:", df.shape)

# 3. Look at the first 5 rows
print(df.head())

# 4. Check for missing values
print("\nMissing values in each column:")
print(df.isnull().sum())


Dataset shape: (489, 12)
            ID team_id    country team_code  \
0  WC-1930_ARG    T-03  Argentina       ARG   
1  WC-1930_BEL    T-06    Belgium       BEL   
2  WC-1930_BOL    T-07    Bolivia       BOL   
3  WC-1930_BRA    T-09     Brazil       BRA   
4  WC-1930_CHL    T-13      Chile       CHL   

                        confederation_name    region_name tournament_id  \
0    South American Football Confederation  South America       WC-1930   
1  Union of European Football Associations         Europe       WC-1930   
2    South American Football Confederation  South America       WC-1930   
3    South American Football Confederation  South America       WC-1930   
4    South American Football Confederation  South America       WC-1930   

             tournament_name  year  matches_played  total_goals stage_reached  
0  1930 FIFA Men's World Cup  1930               5           18         final  
1  1930 FIFA Men's World Cup  1930               2            0   group stage  
2

In [9]:
df['country'].nunique()

85

In [8]:
df['country'].unique()

<ArrowStringArray>
[             'Argentina',                'Belgium',                'Bolivia',
                 'Brazil',                  'Chile',                 'France',
                 'Mexico',               'Paraguay',                   'Peru',
                'Romania',          'United States',                'Uruguay',
             'Yugoslavia',                'Austria',         'Czechoslovakia',
                  'Egypt',                'Germany',                'Hungary',
                  'Italy',            'Netherlands',                  'Spain',
                 'Sweden',            'Switzerland',                   'Cuba',
      'Dutch East Indies',                 'Norway',                 'Poland',
                'England',               'Scotland',            'South Korea',
                 'Turkey',           'West Germany',       'Northern Ireland',
           'Soviet Union',                  'Wales',               'Bulgaria',
               'Colombia',       

In [14]:
# ==============================================================================
# Step 2: Clean the Targets & Understand the Numbers
# ==============================================================================

# Replacing 'final round' with 'final' & 'second group stage' with group stage'
df['stage_reached'] = df['stage_reached'].replace({
    'final round': 'final',
    'second group stage': 'group stage'})
# Clean any accidental spaces and make lowercase
df['stage_reached'] = df['stage_reached'].str.strip().str.lower()

# Map stages from 0 (Group Stage) to 6 (Champion)
stage_map = {
   'group stage': 0,
    'round of 16': 1,
    'quarter-finals': 2,
    'semi-finals': 3,
    'third-place match': 4,
    'final': 5
}
df['stage_number'] = df['stage_reached'].map(stage_map)

# Create a reverse map so we can turn numbers back into words later
number_to_stage = {v: k for k, v in stage_map.items()}

# Quick summary of goals and stages
print("\nAverage goals scored per stage:")
print(df.groupby('stage_reached')['total_goals'].mean().round(2))


Average goals scored per stage:
stage_reached
final                13.57
group stage           2.96
quarter-finals        7.18
round of 16           4.40
semi-finals           7.00
third-place match    11.80
Name: total_goals, dtype: float64


In [15]:
# ==============================================================================
# Step 3: Build Beginner-Friendly Features (Without Cheating!)
# ==============================================================================

# Sort by team and year so history is in the right time order
df = df.sort_values(['team_id', 'year']).reset_index(drop=True)

# Calculate goals per match for historical reference
df['goals_per_match'] = df['total_goals'] / df['matches_played']

# 1. How many World Cups had this team played BEFORE this year?
df['past_appearances'] = df.groupby('team_id').cumcount()

# 2. Average goals scored in PREVIOUS World Cups (.shift(1) ensures we don't peek!)
df['past_avg_goals'] = df.groupby('team_id')['total_goals'].transform(
    lambda x: x.shift(1).expanding().mean()
)

# 3. Average goals per match in PREVIOUS World Cups
df['past_avg_gpm'] = df.groupby('team_id')['goals_per_match'].transform(
    lambda x: x.shift(1).expanding().mean()
)

# 4. Average stage reached in PREVIOUS World Cups
df['past_avg_stage'] = df.groupby('team_id')['stage_number'].transform(
    lambda x: x.shift(1).expanding().mean()
)

# 5. Best stage ever reached before this year
df['past_best_stage'] = df.groupby('team_id')['stage_number'].transform(
    lambda x: x.shift(1).expanding().max()
)

# If a team is playing its VERY FIRST World Cup, its past stats will be NaN (empty)
# Let's fill those empty values with 0 (or the overall average) so the model doesn't crash
df = df.fillna(0)

# Convert text columns (confederation & region) into numbers using One-Hot Encoding
df_encoded = pd.get_dummies(df, columns=['confederation_name', 'region_name'], drop_first=True)

print("Features created successfully! Total columns now:", df_encoded.shape[1])

Features created successfully! Total columns now: 32


In [13]:
df['stage_reached'].unique()

<ArrowStringArray>
[            'final',       'group stage',       'semi-finals',
       'round of 16', 'third-place match',    'quarter-finals']
Length: 6, dtype: str

In [22]:
df_encoded.columns

Index(['ID', 'team_id', 'country', 'team_code', 'tournament_id',
       'tournament_name', 'year', 'matches_played', 'total_goals',
       'stage_reached', 'stage_number', 'goals_per_match', 'past_appearances',
       'past_avg_goals', 'past_avg_gpm', 'past_avg_stage', 'past_best_stage',
       'confederation_name_Confederation of African Football',
       'confederation_name_Confederation of North, Central American and Caribbean Association Football',
       'confederation_name_Oceania Football Confederation',
       'confederation_name_South American Football Confederation',
       'confederation_name_Union of European Football Associations',
       'region_name_Caribbean', 'region_name_Central America',
       'region_name_East Asia', 'region_name_Europe',
       'region_name_Europe, Asia', 'region_name_Middle East',
       'region_name_North America', 'region_name_Oceania',
       'region_name_South America', 'region_name_South East Asia'],
      dtype='str')

In [32]:
df_encoded[['confederation_name_Confederation of African Football',
       'confederation_name_Confederation of North, Central American and Caribbean Association Football',
       'confederation_name_Oceania Football Confederation',
       'confederation_name_South American Football Confederation',
       'confederation_name_Union of European Football Associations',
       'region_name_Caribbean', 'region_name_Central America',
       'region_name_East Asia', 'region_name_Europe',
       'region_name_Europe, Asia', 'region_name_Middle East',
       'region_name_North America', 'region_name_Oceania',
       'region_name_South America', 'region_name_South East Asia']]

,confederation_name_Confederation of African Football,"confederation_name_Confederation of North, Central American and Caribbean Association Football",confederation_name_Oceania Football Confederation,confederation_name_South American Football Confederation,confederation_name_Union of European Football Associations,region_name_Caribbean,region_name_Central America,region_name_East Asia,region_name_Europe,"region_name_Europe, Asia",region_name_Middle East,region_name_North America,region_name_Oceania,region_name_South America,region_name_South East Asia
0,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False
1,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False
2,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False
3,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False
4,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
484,False,False,False,False,True,False,False,False,True,False,False,False,False,False,False
485,False,False,False,False,True,False,False,False,True,False,False,False,False,False,False
486,False,False,False,False,True,False,False,False,True,False,False,False,False,False,False
487,False,False,False,False,True,False,False,False,True,False,False,False,False,False,False


In [33]:
# ==============================================================================
# Step 4: Split Data into Train & Validation (By Year!)
# ==============================================================================

# Pick the columns our model is allowed to use as clues (Features)
ignore_cols = [
    'ID', 'team_id', 'country', 'team_code', 'tournament_id',
    'tournament_name', 'matches_played', 'total_goals',
    'stage_reached', 'stage_number', 'goals_per_match'
]
features = [col for col in df_encoded.columns if col not in ignore_cols]

# Split by year: Train on past tournaments, Validate on 2022
train_data = df_encoded[df_encoded['year'] < 2022].copy()
val_data = df_encoded[df_encoded['year'] == 2022].copy()

X_train = train_data[features]
X_val = val_data[features]

print(f"Training rows (1930-2018): {len(X_train)}")
print(f"Validation rows (2022): {len(X_val)}")


Training rows (1930-2018): 457
Validation rows (2022): 32


In [38]:
# ==============================================================================
# Step 5: Problem 1 — Predict total_goals (Regression)
# ==============================================================================

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error

y_train_goals = train_data['total_goals']
y_val_goals = val_data['total_goals']

# 1. Create and train the model
goal_model = RandomForestRegressor(n_estimators=100, max_depth=6, random_state=42)
goal_model.fit(X_train, y_train_goals)

# 2. Make predictions on 2022
pred_goals = goal_model.predict(X_val)

# 3. Check how close we were (RMSE)
rmse = root_mean_squared_error(y_val_goals, pred_goals)
print(f"Goals Prediction RMSE on 2022: {rmse:.2f} goals")


Goals Prediction RMSE on 2022: 3.55 goals


In [44]:
from sklearn.metrics import root_mean_squared_error
baseline = [y_train_goals.mean()] * len(y_val_goals)
print(root_mean_squared_error(y_val_goals, baseline))

4.026421758701227


In [45]:
features = X_train.columns.tolist()

for yr in [2002, 2006, 2010, 2014, 2018, 2022]:
    tr = df_encoded[df_encoded['year'] < yr]
    va = df_encoded[df_encoded['year'] == yr]

    model = RandomForestRegressor(n_estimators=100, max_depth=6, random_state=42)
    model.fit(tr[features], tr['total_goals'])

    model_rmse = root_mean_squared_error(va['total_goals'], model.predict(va[features]))
    base_rmse = root_mean_squared_error(va['total_goals'], [tr['total_goals'].mean()] * len(va))
    print(yr, round(model_rmse, 2), round(base_rmse, 2))

2002 3.82 3.98
2006 2.95 3.65
2010 2.92 3.83
2014 3.35 3.98
2018 4.06 4.01
2022 3.55 4.03


In [46]:
features = X_train.columns.tolist()
results = []

for yr in [2002, 2006, 2010, 2014, 2018, 2022]:
    tr = df_encoded[df_encoded['year'] < yr]
    va = df_encoded[df_encoded['year'] == yr]

    model = RandomForestRegressor(n_estimators=100, max_depth=6, random_state=42)
    model.fit(tr[features], tr['total_goals'])

    model_rmse = root_mean_squared_error(va['total_goals'], model.predict(va[features]))
    base_rmse = root_mean_squared_error(va['total_goals'], [tr['total_goals'].mean()] * len(va))

    # decide the wording for the Result column
    if model_rmse < base_rmse:
        gain = (base_rmse - model_rmse) / base_rmse * 100
        result = f"model better by {gain:.1f}%"
    else:
        result = "baseline better"

    # save this year's row
    results.append({
        'Validation year': yr,
        'Model RMSE': round(model_rmse, 2),
        'Baseline RMSE': round(base_rmse, 2),
        'Result': result
    })

# turn the list of rows into a table
results_df = pd.DataFrame(results)

# make the whole row bold in years where the baseline wins
def bold_if_baseline_wins(row):
    weight = 'font-weight: bold' if row['Result'] == 'baseline better' else ''
    return [weight] * len(row)

results_df.style.apply(bold_if_baseline_wins, axis=1).hide(axis='index')

Validation year,Model RMSE,Baseline RMSE,Result
2002,3.820000,3.980000,model better by 3.8%
2006,2.950000,3.650000,model better by 19.3%
2010,2.920000,3.830000,model better by 23.8%
2014,3.350000,3.980000,model better by 15.8%
2018,4.060000,4.010000,baseline better
2022,3.550000,4.030000,model better by 11.8%


What this tells you

- The model beats the baseline in 5 of 6 tournaments, so it has found some real signal, but not consistently. In 2018 it is slightly worse than guessing the average (4.06 against 4.01), and in 2002 the gain is tiny.

- The size of the improvement varies a lot, from 24% in 2010 to 4% in 2002. That is why a single-year result like 3.55 should not be reported as "the model is 12% better" without this context.

- A reasonable way to report it is: "The model improved on the baseline in most tournaments, with an average gain of roughly 12% in RMSE, but not in every one, and 2018 was slightly worse." (The average of the six gains above is about 12%.)

In [47]:
# ==============================================================================
# Step 6: Problem 2 — Predict stage_reached (Classification)
# ==============================================================================

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

y_train_stage = train_data['stage_number']
y_val_stage = val_data['stage_number']

# 1. Create and train the classifier
stage_model = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
stage_model.fit(X_train, y_train_stage)

# 2. Predict stage numbers for 2022
pred_stage_num = stage_model.predict(X_val)

# 3. Check F1 Score
f1 = f1_score(y_val_stage, pred_stage_num, average='weighted')
print(f"Stage Prediction F1 Score on 2022: {f1:.2f}")


Stage Prediction F1 Score on 2022: 0.55


In [48]:
# ==============================================================================
# Step 7: See Your Results Side-by-Side!
# ==============================================================================

# Create a clean table comparing Actual vs Predicted for 2022
results = pd.DataFrame({
    'Country': val_data['country'],
    'Actual_Goals': y_val_goals,
    'Predicted_Goals': np.round(pred_goals, 1),
    'Actual_Stage': y_val_stage.map(number_to_stage),
    'Predicted_Stage': pd.Series(pred_stage_num, index=val_data.index).map(number_to_stage)
})

# Show top 15 teams sorted by actual goals
print("\n--- 2022 WORLD CUP: ACTUAL VS PREDICTED ---")
print(results.sort_values('Actual_Goals', ascending=False).head(15).to_string(index=False))



--- 2022 WORLD CUP: ACTUAL VS PREDICTED ---
    Country  Actual_Goals  Predicted_Goals      Actual_Stage Predicted_Stage
     France            16              7.3             final     group stage
  Argentina            15              7.3             final  quarter-finals
    England            13              7.8    quarter-finals  quarter-finals
   Portugal            12              6.0    quarter-finals     group stage
Netherlands            10              7.7    quarter-finals     group stage
      Spain             9              7.6       round of 16     round of 16
    Croatia             8              6.8 third-place match     group stage
     Brazil             8             10.0    quarter-finals  quarter-finals
    Morocco             6              3.5 third-place match     group stage
    Germany             6             11.8       group stage     group stage
South Korea             5              4.3       round of 16     group stage
Switzerland             5      

In [49]:
results['Goal_Error'] = results['Predicted_Goals'] - results['Actual_Goals']
print(results.sort_values('Country').to_string(index=False))

      Country  Actual_Goals  Predicted_Goals      Actual_Stage Predicted_Stage  Goal_Error
    Argentina            15              7.3             final  quarter-finals        -7.7
    Australia             4              4.1       round of 16     group stage         0.1
      Belgium             1              8.4       group stage     round of 16         7.4
       Brazil             8             10.0    quarter-finals  quarter-finals         2.0
     Cameroon             4              4.1       group stage     group stage         0.1
       Canada             2              2.5       group stage     group stage         0.5
   Costa Rica             3              3.6       group stage     group stage         0.6
      Croatia             8              6.8 third-place match     group stage        -1.2
      Denmark             1              4.9       group stage     group stage         3.9
      Ecuador             4              4.4       group stage     group stage         0.4